In [1]:
from dotenv import load_dotenv

load_dotenv()

True

## Summarize messages

In [2]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents.middleware import SummarizationMiddleware

agent = create_agent(
    model="ollama:gemma4:e4b",
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="ollama:gemma4:e4b",
            trigger=("tokens", 100),
            keep=("messages", 1)
        )
    ],
)

In [3]:
from langchain.messages import HumanMessage, AIMessage
from pprint import pprint

response = agent.invoke(
    {"messages": [
        HumanMessage(content="What is the capital of the moon?"),
        AIMessage(content="The capital of the moon is Lunapolis."),
        HumanMessage(content="What is the weather in Lunapolis?"),
        AIMessage(content="Skies are clear, with a high of 120C and a low of -100C."),
        HumanMessage(content="How many cheese miners live in Lunapolis?"),
        AIMessage(content="There are 100,000 cheese miners living in Lunapolis."),
        HumanMessage(content="Do you think the cheese miners' union will strike?"),
        AIMessage(content="Yes, because they are unhappy with the new president."),
        HumanMessage(content="If you were Lunapolis' new president how would you respond to the cheese miners' union?"),
        ]},
    {"configurable": {"thread_id": "1"}}
)

pprint(response)

{'messages': [HumanMessage(content='Here is a summary of the conversation to date:\n\n## SESSION INTENT\nThe user\'s primary goal is to gather specific, descriptive, and potentially fictional information about a location called "Lunapolis," which is identified as the capital of the moon.\n\n## SUMMARY\nKey facts established during the session about Lunapolis:\n*   **Capital:** Lunapolis.\n*   **Climate/Weather:** Clear skies, with a high of 120C and a low of -100C.\n*   **Population Detail:** There are 100,000 cheese miners living in the city.\n*   **Social Prediction:** The cheese miners\' union is predicted to strike due to discontent with the new president.\n\n## ARTIFACTS\nNone\n\n## NEXT STEPS\nNone', additional_kwargs={'lc_source': 'summarization'}, response_metadata={}, id='50450064-35f4-47c1-8f11-e7e6417151d7'),
              HumanMessage(content="If you were Lunapolis' new president how would you respond to the cheese miners' union?", additional_kwargs={}, response_metadata={}

In [4]:
print(response["messages"][0].content)

Here is a summary of the conversation to date:

## SESSION INTENT
The user's primary goal is to gather specific, descriptive, and potentially fictional information about a location called "Lunapolis," which is identified as the capital of the moon.

## SUMMARY
Key facts established during the session about Lunapolis:
*   **Capital:** Lunapolis.
*   **Climate/Weather:** Clear skies, with a high of 120C and a low of -100C.
*   **Population Detail:** There are 100,000 cheese miners living in the city.
*   **Social Prediction:** The cheese miners' union is predicted to strike due to discontent with the new president.

## ARTIFACTS
None

## NEXT STEPS
None


## Trim/delete messages

In [5]:
from typing import Any
from langchain.agents import AgentState
from langchain.messages import RemoveMessage
from langgraph.runtime import Runtime
from langchain.agents.middleware import before_agent
from langchain.messages import ToolMessage

@before_agent
def trim_messages(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    """Remove all the tool messages from the state"""
    messages = state["messages"]

    tool_messages = [m for m in messages if isinstance(m, ToolMessage)]
    
    return {"messages": [RemoveMessage(id=m.id) for m in tool_messages]}

In [6]:
agent = create_agent(
    model="ollama:gemma4:e4b",
    checkpointer=InMemorySaver(),
    middleware=[trim_messages],
)

In [7]:
response = agent.invoke(
    {"messages": [
        HumanMessage(content="My device won't turn on. What should I do?"),
        ToolMessage(content="blorp-x7 initiating diagnostic ping…", tool_call_id="1"),
        AIMessage(content="Is the device plugged in and turned on?"),
        HumanMessage(content="Yes, it's plugged in and turned on."),
        ToolMessage(content="temp=42C voltage=2.9v … greeble complete.", tool_call_id="2"),
        AIMessage(content="Is the device showing any lights or indicators?"),
        HumanMessage(content="What's the temperature of the device?")
        ]},
    {"configurable": {"thread_id": "2"}}
)

pprint(response)

{'messages': [HumanMessage(content="My device won't turn on. What should I do?", additional_kwargs={}, response_metadata={}, id='88618c2c-4979-44c5-95d0-89d5b3dc302f'),
              AIMessage(content='Is the device plugged in and turned on?', additional_kwargs={}, response_metadata={}, id='dfe9ca90-97f4-4cfa-bc70-4aaf68323fc9', tool_calls=[], invalid_tool_calls=[]),
              HumanMessage(content="Yes, it's plugged in and turned on.", additional_kwargs={}, response_metadata={}, id='c88f7ece-473f-447e-b651-cd472bf1ebcb'),
              AIMessage(content='Is the device showing any lights or indicators?', additional_kwargs={}, response_metadata={}, id='fb7de6db-7c81-457b-9c11-91f99cdf1ca5', tool_calls=[], invalid_tool_calls=[]),
              HumanMessage(content="What's the temperature of the device?", additional_kwargs={}, response_metadata={}, id='d744a979-47f1-45f1-99d7-3ccb20d02be1'),
              AIMessage(content="I cannot physically check the temperature of your device, as I

In [8]:
print(response["messages"][-1].content)

I cannot physically check the temperature of your device, as I am an AI. You will need to check that yourself.

**Could you please tell me if the device feels:**

1.  **Hot/Warm:** (If it's hot, it might be overheating, which could be a safety issue or a major system error.)
2.  **Cold/Room Temperature:** (If it's cold, it might just mean it hasn't been drawing power.)

Knowing the temperature will help us determine if overheating is the issue, or if the problem is related to power delivery.
